### NPD loc.parquet generation

In [76]:
import pandas as pd
import os
from pathlib import Path
import numpy as np
from sklearn.model_selection import train_test_split

from fastai.learner import *
from fastai.metrics import accuracy
import time

from tsai.all import TSDatasets, TSDataLoaders, TSStandardize, Categorize, combine_split_data

import torch
from torch.utils.data import Dataset, DataLoader

from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

In [77]:

# Paths
ROOT_PATH = Path().resolve().parent  

DATA_NS_PATH = ROOT_PATH / "data" / "Data_NS"

NPD_WELLLOGS_DIR = DATA_NS_PATH / "npd_welllogs"
METADATA_PATH = DATA_NS_PATH / "npd_metadata" / "tops_conditioned_corr.csv"
OUTPUT_PARQUET = DATA_NS_PATH / "loc.parquet"
COMBINED_PARQUET = DATA_NS_PATH / "npd_welllogs_combined.parquet"


In [78]:
print('Reading metadata...')
df_tops = pd.read_csv(METADATA_PATH, usecols=['wellbore_name', 'latitude', 'longitude'])

print('Path to well logs:', NPD_WELLLOGS_DIR)
print('Path to metadata:', METADATA_PATH)
print('Output path for Parquet file:', OUTPUT_PARQUET)

Reading metadata...
Path to well logs: /Users/enorabarbier/Documents/1.M2 Courses/CRP/CRP_SLB/datasets/north_sea/data/Data_NS/npd_welllogs
Path to metadata: /Users/enorabarbier/Documents/1.M2 Courses/CRP/CRP_SLB/datasets/north_sea/data/Data_NS/npd_metadata/tops_conditioned_corr.csv
Output path for Parquet file: /Users/enorabarbier/Documents/1.M2 Courses/CRP/CRP_SLB/datasets/north_sea/data/Data_NS/loc.parquet


In [79]:
# Unique locations
df_locs = df_tops.drop_duplicates(subset=['wellbore_name']).set_index('wellbore_name')

# Rename columns/index to match schema
df_locs = df_locs.rename(columns={'latitude': 'Latitude', 'longitude': 'Longitude'})
df_locs.index.name = 'well_name'

# Wells available as parquet files
parquet_files = [
    f for f in os.listdir(NPD_WELLLOGS_DIR)
    if f.endswith('.parquet') and f != 'loc.parquet'
]
well_names = [Path(f).stem for f in parquet_files]

# Filter to wells with parquet logs
df_locs_filtered = df_locs.loc[df_locs.index.intersection(well_names)]

# Save
df_locs_filtered.to_parquet(OUTPUT_PARQUET)
print(f"Saved loc.parquet with {len(df_locs_filtered)} wells to {OUTPUT_PARQUET}")
display(df_locs_filtered.head())


Saved loc.parquet with 1377 wells to /Users/enorabarbier/Documents/1.M2 Courses/CRP/CRP_SLB/datasets/north_sea/data/Data_NS/loc.parquet


,Latitude,Longitude
well_name,,
1_2_1,56.89,2.48
1_2_2,56.99,2.50
1_3_1,56.86,2.85
1_3_10,56.93,2.71
1_3_10_A,56.93,2.71


### NPD Tops.parquet generation

In [80]:
# Load NPD top train data
df_tops = pd.read_csv(METADATA_PATH)

# Pivot to create columns for each top containing their depths
pivoted_tops = df_tops.pivot_table(index='wellbore_name', columns='valid_marker_name', values='depth')

# Rename the index to match the structure of tops.parquet
pivoted_tops.index.name = 'well_name'

# Save the structured dataframe to npd_tops.parquet
pivoted_tops.to_parquet('npd_tops.parquet')
print('npd_tops.parquet successfully created with shape:', pivoted_tops.shape)


npd_tops.parquet successfully created with shape: (1709, 6)


In [81]:
well= pd.read_parquet('npd_tops.parquet', engine='fastparquet')
well.head()

valid_marker_name,DUNLIN,HORDALAND,ROGALAND,SHETLAND,VESTLAND,VIKING
well_name,,,,,,
10_4_1,NaN,751.31236,1607.61160,1935.69560,7460.63016,NaN
10_5_1,NaN,623.35960,885.82680,1099.08140,4829.39648,NaN
10_7_1,NaN,1479.65884,2139.10768,2562.33604,5049.21276,NaN
10_8_1,NaN,721.78480,1692.91344,2152.23104,NaN,NaN
11_10_1,NaN,1410.76120,3356.29932,3438.32032,NaN,NaN


In [82]:
well.shape

(1709, 6)

In [83]:
# ---------------- Paths ----------------
OUT_LOC = DATA_NS_PATH / "loc.parquet"
OUT_TOPS = DATA_NS_PATH / "npd_tops.parquet"
OUT_MERGED = DATA_NS_PATH / "npd_tops_with_loc.parquet"


In [84]:
# ---------------- Config ----------------
target_tops = ["HORDALAND", "ROGALAND", "SHETLAND", "VIKING", "VESTLAND", "DUNLIN"]

# ---------------- Load once ----------------
df_raw = pd.read_csv(METADATA_PATH)


In [85]:

# ---------------- Keep only wells that have a parquet welllog ----------------
parquet_wells = {f.stem for f in NPD_WELLLOGS_DIR.glob("*.parquet")} - {"loc"}
# If your parquet filenames match df_raw["wellbore_name"], this will work directly.
# If not, we can normalize names, but try this first.

# ---------------- Build loc table ----------------
df_loc = (
    df_raw.loc[:, ["wellbore_name", "latitude", "longitude"]]
      .drop_duplicates(subset=["wellbore_name"])
      .set_index("wellbore_name")
      .rename(columns={"latitude": "Latitude", "longitude": "Longitude"})
)
df_loc.index.name = "well_name"

# filter to wells we actually have logs for
df_loc = df_loc.loc[df_loc.index.intersection(parquet_wells)]

# ---------------- Filter tops then pivot ----------------
df_tops = df_raw[
    (df_raw["marker_name"].isin(target_tops)) &
    (df_raw["outlier"] != -1) &
    (df_raw["valid_presence"] == True)
].copy()

# If the correct column to pivot on is valid_marker_name, keep it.
# Otherwise, use marker_name. Pick ONE consistently.
pivot_col = "valid_marker_name" if "valid_marker_name" in df_tops.columns else "marker_name"

df_tops_pivot = (
    df_tops.pivot_table(
        index="wellbore_name",
        columns=pivot_col,
        values="depth",
        aggfunc="first"   # in case duplicates exist
    )
)
df_tops_pivot.index.name = "well_name"

# filter to wells we actually have logs for
df_tops_pivot = df_tops_pivot.loc[df_tops_pivot.index.intersection(parquet_wells)]

# ---------------- Save the two parquet files (optional) ----------------
df_loc.to_parquet(OUT_LOC)
df_tops_pivot.to_parquet(OUT_TOPS)

# ---------------- Merge them into one parquet ----------------
# Join on index (well_name). "left" keeps only wells that have tops.
df_merged = df_tops_pivot.join(df_loc, how="left")

# (optional) if you want to keep ALL wells that have logs, even if no tops:
# df_merged = df_loc.join(df_tops_pivot, how="left")

df_merged.to_parquet(OUT_MERGED)

print("loc:", df_loc.shape)
print("tops:", df_tops_pivot.shape)
print("merged:", df_merged.shape)
display(df_merged.head())


loc: (1377, 2)
tops: (1324, 6)
merged: (1324, 8)


,DUNLIN,HORDALAND,ROGALAND,SHETLAND,VESTLAND,VIKING,Latitude,Longitude
well_name,,,,,,,,
10_4_1,NaN,751.31236,1607.61160,1935.69560,7460.63016,NaN,57.68,5.16
10_5_1,NaN,623.35960,885.82680,1099.08140,4829.39648,NaN,57.58,5.59
10_7_1,NaN,1479.65884,2139.10768,2562.33604,5049.21276,NaN,57.47,5.06
10_8_1,NaN,721.78480,1692.91344,NaN,NaN,NaN,57.42,5.57
11_10_1,NaN,1410.76120,3356.29932,3438.32032,NaN,NaN,57.01,6.17


In [86]:

# ---------------- Keep only wells that have a parquet welllog ----------------
parquet_wells = {f.stem for f in NPD_WELLLOGS_DIR.glob("*.parquet")} - {"loc"}
# If your parquet filenames match df_raw["wellbore_name"], this will work directly.
# If not, we can normalize names, but try this first.

# ---------------- Build loc table ----------------
df_loc = (
    df_raw.loc[:, ["wellbore_name", "latitude", "longitude"]]
      .drop_duplicates(subset=["wellbore_name"])
      .set_index("wellbore_name")
      .rename(columns={"latitude": "Latitude", "longitude": "Longitude"})
)
df_loc.index.name = "well_name"

# filter to wells we actually have logs for
df_loc = df_loc.loc[df_loc.index.intersection(parquet_wells)]

# ---------------- Filter tops then pivot ----------------
df_tops = df_raw[
    (df_raw["marker_name"].isin(target_tops)) &
    (df_raw["outlier"] != -1) &
    (df_raw["valid_presence"] == True)
].copy()

# If the correct column to pivot on is valid_marker_name, keep it.
# Otherwise, use marker_name. Pick ONE consistently.
pivot_col = "valid_marker_name" if "valid_marker_name" in df_tops.columns else "marker_name"

df_tops_pivot = (
    df_tops.pivot_table(
        index="wellbore_name",
        columns=pivot_col,
        values="depth",
        aggfunc="first"   # in case duplicates exist
    )
)
df_tops_pivot.index.name = "well_name"

# filter to wells we actually have logs for
df_tops_pivot = df_tops_pivot.loc[df_tops_pivot.index.intersection(parquet_wells)]

# ---------------- Save the two parquet files (optional) ----------------
df_loc.to_parquet(OUT_LOC)
df_tops_pivot.to_parquet(OUT_TOPS)

# ---------------- Merge them into one parquet ----------------
# Join on index (well_name). "left" keeps only wells that have tops.
df_merged = df_tops_pivot.join(df_loc, how="left")

# (optional) if you want to keep ALL wells that have logs, even if no tops:
# df_merged = df_loc.join(df_tops_pivot, how="left")

df_merged.to_parquet(OUT_MERGED)

print("loc:", df_loc.shape)
print("tops:", df_tops_pivot.shape)
print("merged:", df_merged.shape)
display(df_merged.head())


loc: (1377, 2)
tops: (1324, 6)
merged: (1324, 8)


,DUNLIN,HORDALAND,ROGALAND,SHETLAND,VESTLAND,VIKING,Latitude,Longitude
well_name,,,,,,,,
10_4_1,NaN,751.31236,1607.61160,1935.69560,7460.63016,NaN,57.68,5.16
10_5_1,NaN,623.35960,885.82680,1099.08140,4829.39648,NaN,57.58,5.59
10_7_1,NaN,1479.65884,2139.10768,2562.33604,5049.21276,NaN,57.47,5.06
10_8_1,NaN,721.78480,1692.91344,NaN,NaN,NaN,57.42,5.57
11_10_1,NaN,1410.76120,3356.29932,3438.32032,NaN,NaN,57.01,6.17


In [87]:
df_logs = pd.read_parquet(COMBINED_PARQUET)
df_logs.head(20)

,GR_CLEAN,RHOB_CLEAN,DTC_CLEAN,DEEPRES_CLEAN,NEUT_CLEAN,DS_INDEX,well_name
0,97.027901,NaN,NaN,NaN,NaN,383.660099,30_10_3
1,96.591103,NaN,NaN,NaN,NaN,384.160099,30_10_3
2,96.081902,NaN,NaN,NaN,NaN,384.660099,30_10_3
3,94.928596,NaN,NaN,NaN,NaN,385.160099,30_10_3
4,93.775299,NaN,NaN,NaN,NaN,385.660099,30_10_3
5,92.273102,NaN,NaN,NaN,NaN,386.160099,30_10_3
6,88.713600,NaN,NaN,NaN,NaN,386.660099,30_10_3
7,85.153999,NaN,NaN,NaN,NaN,387.160099,30_10_3
8,75.526497,NaN,NaN,NaN,NaN,387.660099,30_10_3
9,61.999901,NaN,NaN,NaN,NaN,388.160099,30_10_3


In [88]:
#print first non nan in deepres_clean
first_non_nan_ln = df_logs['DEEPRES_CLEAN'].dropna().iloc[0]

In [89]:
#correct ln to through exponential log10 transformation on DEEPRES_CLEAN
if 'DEEPRES_CLEAN' in df_logs.columns:
    df_logs['DEEPRES_CLEAN'] = np.exp(df_logs['DEEPRES_CLEAN'])
    df_logs['DEEPRES_CLEAN'] = np.log10(df_logs['DEEPRES_CLEAN'])

In [90]:
first_non_nan_log = df_logs['DEEPRES_CLEAN'].dropna().iloc[0]
print("First non-NaN value in DEEPRES_CLEAN before vs after transformation:", first_non_nan_ln, first_non_nan_log)

First non-NaN value in DEEPRES_CLEAN before vs after transformation: -0.12261973198317062 -0.05325307297274669


In [91]:
# Min-max normalization applied column-wise using fixed physical bounds
# DEEPRES_log10 refers to DEEPRES_CLEAN after the log10 transformation above
min_max_dict = {
    "GR_CLEAN":       [0, 200],
    "RHOB_CLEAN":     [1.95, 2.95],
    "NEUT_CLEAN":     [-0.15, 0.7],
    "DTC_CLEAN":      [40, 240],
    "DEEPRES_log10":  [np.log10(0.1), np.log10(2000)],
}

col_rename = {"DEEPRES_log10": "DEEPRES_CLEAN"}  # maps dict key -> actual df column

for key, (lo, hi) in min_max_dict.items():
    col = col_rename.get(key, key)
    if col in df_logs.columns:
        df_logs[col] = ((df_logs[col] - lo) / (hi - lo)).clip(0, 1)     

print("Normalization applied. Sample statistics:")
display(df_logs[["GR_CLEAN", "RHOB_CLEAN", "NEUT_CLEAN", "DTC_CLEAN", "DEEPRES_CLEAN"]].describe())

Normalization applied. Sample statistics:


,GR_CLEAN,RHOB_CLEAN,NEUT_CLEAN,DTC_CLEAN,DEEPRES_CLEAN
count,2.884073e+07,1.486865e+07,1.204201e+07,1.505192e+07,2.718266e+07
mean,3.391687e-01,3.688038e-01,5.582074e-01,3.916890e-01,2.769589e-01
std,1.659729e-01,2.139188e-01,1.620238e-01,1.508154e-01,9.285517e-02
min,0.000000e+00,0.000000e+00,1.800000e-02,1.000000e-01,5.935149e-02
25%,2.237545e-01,1.850000e-01,4.474091e-01,2.630631e-01,2.182081e-01
50%,3.287855e-01,3.810001e-01,5.616471e-01,3.944521e-01,2.657829e-01
75%,4.361000e-01,5.481000e-01,6.707175e-01,5.193910e-01,3.163941e-01
max,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00


In [92]:
# Ensure consistent naming
df_logs.rename(columns={"well_name": "well_name"}, inplace=True)
df_logs["well_name"] = df_logs["well_name"].astype(str)
df_merged.index = df_merged.index.astype(str)

# Merge (left join keeps all depth rows)
df_final = df_logs.merge(
    df_merged,
    on="well_name",
    how="left"
)

print("Final shape:", df_final.shape)
display(df_final.head())

Final shape: (31236076, 15)


,GR_CLEAN,RHOB_CLEAN,DTC_CLEAN,DEEPRES_CLEAN,NEUT_CLEAN,DS_INDEX,well_name,DUNLIN,HORDALAND,ROGALAND,SHETLAND,VESTLAND,VIKING,Latitude,Longitude
0,0.485140,NaN,NaN,NaN,NaN,383.660099,30_10_3,NaN,NaN,7099.73776,NaN,NaN,NaN,60.05,2.17
1,0.482956,NaN,NaN,NaN,NaN,384.160099,30_10_3,NaN,NaN,7099.73776,NaN,NaN,NaN,60.05,2.17
2,0.480410,NaN,NaN,NaN,NaN,384.660099,30_10_3,NaN,NaN,7099.73776,NaN,NaN,NaN,60.05,2.17
3,0.474643,NaN,NaN,NaN,NaN,385.160099,30_10_3,NaN,NaN,7099.73776,NaN,NaN,NaN,60.05,2.17
4,0.468876,NaN,NaN,NaN,NaN,385.660099,30_10_3,NaN,NaN,7099.73776,NaN,NaN,NaN,60.05,2.17


In [93]:
WELL_COL = "well_name"
DEPTH_COL = "DS_INDEX"   # your depth axis
MARKERS = ["HORDALAND", "ROGALAND", "SHETLAND", "VIKING", "VESTLAND", "DUNLIN"]

# (a) per-well top depths (since repeated)
tops_wide = df_final[[WELL_COL] + MARKERS].groupby(WELL_COL).first().reset_index()


In [94]:
tops_wide.head(20)

,well_name,HORDALAND,ROGALAND,SHETLAND,VIKING,VESTLAND,DUNLIN
0,10_4_1,751.31236,1607.61160,1935.69560,NaN,7460.63016,NaN
1,10_5_1,623.35960,885.82680,1099.08140,NaN,4829.39648,NaN
2,10_7_1,1479.65884,2139.10768,2562.33604,NaN,5049.21276,NaN
3,10_8_1,721.78480,1692.91344,NaN,NaN,NaN,NaN
4,11_10_1,1410.76120,3356.29932,3438.32032,NaN,NaN,NaN
5,11_5_1,NaN,NaN,1164.69820,NaN,4186.35184,NaN
6,15_12_1,4189.63268,7857.61180,8671.26012,9862.20504,10246.06332,NaN
7,15_12_10_S,4619.42272,8366.14200,9074.80344,10698.81924,11220.47280,NaN
8,15_12_11_S,4340.55132,8300.52520,9317.58560,10633.20244,11138.45180,NaN
9,15_12_12,4084.64580,7342.51992,8021.65380,9163.38612,NaN,NaN


In [95]:
# (b) long format: one row = one (well, marker, top_depth)
tops = (tops_wide
        .melt(id_vars=WELL_COL, value_vars=MARKERS,
              var_name="marker_name", value_name="top_depth")
        .dropna(subset=["top_depth"])
        .sort_values([WELL_COL, "top_depth"])
        .reset_index(drop=True))

In [96]:
tops.head(20)

,well_name,marker_name,top_depth
0,10_4_1,HORDALAND,751.31236
1,10_4_1,ROGALAND,1607.61160
2,10_4_1,SHETLAND,1935.69560
3,10_4_1,VESTLAND,7460.63016
4,10_5_1,HORDALAND,623.35960
5,10_5_1,ROGALAND,885.82680
6,10_5_1,SHETLAND,1099.08140
7,10_5_1,VESTLAND,4829.39648
8,10_7_1,HORDALAND,1479.65884
9,10_7_1,ROGALAND,2139.10768


In [97]:
#%pip install iterative-stratification


In [98]:
# Build train/val/test splits based on wells, stratified by marker presence (multi-label)
# build multi-hot label matrix per well
well_marker = (tops.assign(v=1)
               .pivot_table(index=WELL_COL, columns="marker_name", values="v",
                            fill_value=0, aggfunc="max")
               .astype(int))

# Prepare data for stratified split
wells = well_marker.index.to_numpy()
Y = well_marker.to_numpy()

# Stratified split parameters
test_size = 0.2
random_state = 42
n_splits = 1
test_size = 0.20
val_size  = 0.10 

# Stratified split
msss1 = MultilabelStratifiedShuffleSplit(test_size=test_size, random_state=random_state, n_splits=n_splits)
trainval_idx, test_idx = next(msss1.split(wells, Y))

wells_trainval = wells[trainval_idx]
Y_trainval = Y[trainval_idx]

wells_test = wells[test_idx]

val_size_within_trainval = val_size / (1.0 - test_size)

msss2 = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=val_size_within_trainval, random_state=random_state)
train_idx2, val_idx = next(msss2.split(wells_trainval, Y_trainval))

train_wells = set(wells_trainval[train_idx2])
val_wells   = set(wells_trainval[val_idx])
test_wells  = set(wells_test)

In [99]:
import numpy as np
import pandas as pd

def marker_freq(well_set, well_marker):
    """
    Compute marker counts and ratios for a given set of wells.

    Parameters:
    - well_set: set of well names to compute frequencies for
    - well_marker: DataFrame (index=well_name, columns=marker_name) multi-hot matrix

    Returns:
    - DataFrame with marker counts and ratios for the specified wells
    """
    well_sum = well_marker.loc[list(well_set)].sum()
    total = well_sum.sum()
    well_ratio = well_sum / total if total > 0 else well_sum * np.nan
    return pd.concat([well_sum, well_ratio], axis=1, keys=["count", "ratio"])


def check_splits(train_wells, val_wells, test_wells, well_marker, rtol=0.1):
    # ---- tables ----
    train_tab = marker_freq(train_wells, well_marker)
    val_tab   = marker_freq(val_wells, well_marker)
    test_tab  = marker_freq(test_wells, well_marker)

    # ---- overlap checks ----
    ov_tv = train_wells & val_wells
    ov_tt = train_wells & test_wells
    ov_vt = val_wells & test_wells

    if len(ov_tv) == 0 and len(ov_tt) == 0 and len(ov_vt) == 0:
        print("\nNo overlap between train/val/test wells")
    else:
        print("\nAttention overlap, fix split!")
        if ov_tv: print("train ∩ val:", sorted(list(ov_tv))[:20], "..." if len(ov_tv) > 20 else "")
        if ov_tt: print("train ∩ test:", sorted(list(ov_tt))[:20], "..." if len(ov_tt) > 20 else "")
        if ov_vt: print("val ∩ test:", sorted(list(ov_vt))[:20], "..." if len(ov_vt) > 20 else "")

    # ---- ratio closeness checks ----
    # compare ratios pairwise (train vs val, train vs test, val vs test)
    close_tv = np.isclose(train_tab["ratio"], val_tab["ratio"], equal_nan=True, rtol=rtol).all()
    close_tt = np.isclose(train_tab["ratio"], test_tab["ratio"], equal_nan=True, rtol=rtol).all()
    close_vt = np.isclose(val_tab["ratio"], test_tab["ratio"], equal_nan=True, rtol=rtol).all()

    print(f"\nRatios close (rtol={rtol}):")
    print("  train vs val :", close_tv)
    print("  train vs test:", close_tt)
    print("  val vs test  :", close_vt)

    # ---- counts ----
    print("\nSplit sizes (#wells):", len(train_wells), len(val_wells), len(test_wells))

    # ---- print tables ----
    print("\nTrain marker split:\n", train_tab)
    print("\nVal marker split:\n", val_tab)
    print("\nTest marker split:\n", test_tab)

    return train_tab, val_tab, test_tab

In [100]:
# ---------------------------------------------------------------------------
# 1. Filter wells with >= 65% of tops present
# ---------------------------------------------------------------------------
MIN_TOP_FRACTION = 0.65
n_markers = len(MARKERS)
min_tops_required = int(np.ceil(MIN_TOP_FRACTION * n_markers))  # ceil(0.65 * 6) = 4

# tops_wide has one row per well, columns = MARKERS, values = top depths or NaN
tops_wide_indexed = tops_wide.set_index(WELL_COL)
tops_present_count = tops_wide_indexed[MARKERS].notna().sum(axis=1)

valid_wells = set(tops_present_count[tops_present_count >= min_tops_required].index)
print(f"Wells with >= {min_tops_required}/{n_markers} tops: {len(valid_wells)} / {len(tops_wide_indexed)}")

Wells with >= 4/6 tops: 620 / 1536


In [101]:
df_tops = tops_wide_indexed[MARKERS]

In [102]:
# ---------------------------------------------------------------------------
# 3. Build df_log: just the columns the pipeline needs
# ---------------------------------------------------------------------------
LOG_COLS = ["GR_CLEAN", "RHOB_CLEAN", "DEEPRES_CLEAN"]

df_log = df_final[[WELL_COL, DEPTH_COL] + LOG_COLS].copy()

In [103]:
WELL_COL = "well_name"
DEPTH_COL = "DS_INDEX"
MARKERS = ["HORDALAND", "ROGALAND", "SHETLAND", "VIKING", "VESTLAND", "DUNLIN"]
MIN_TOP_FRACTION = 0.65
n_markers = len(MARKERS)
min_tops_required = int(np.ceil(MIN_TOP_FRACTION * n_markers))  # = 4

# Per-well top depths
tops_wide = df_final[[WELL_COL] + MARKERS].groupby(WELL_COL).first().reset_index()
tops_wide_indexed = tops_wide.set_index(WELL_COL)

# Filter: keep only wells with >= 4 tops
tops_present_count = tops_wide_indexed[MARKERS].notna().sum(axis=1)
valid_wells = tops_present_count[tops_present_count >= min_tops_required].index
tops_filtered = tops_wide_indexed.loc[valid_wells]

print(f"Wells with >= {min_tops_required}/{n_markers} tops: {len(tops_filtered)} / {len(tops_wide_indexed)}")

# Also need wells that exist in the logs
log_wells = set(df_final[WELL_COL].unique())
tops_filtered = tops_filtered.loc[tops_filtered.index.isin(log_wells)]
print(f"Wells also present in logs: {len(tops_filtered)}")

Wells with >= 4/6 tops: 620 / 1536
Wells also present in logs: 620


In [104]:
# Multi-hot: which markers does each well have?
well_marker = tops_filtered[MARKERS].notna().astype(int)

wells = well_marker.index.to_numpy()
Y = well_marker.to_numpy()

print(f"Splitting {len(wells)} wells with marker distribution:")
print(well_marker.sum().to_string())

Splitting 620 wells with marker distribution:
HORDALAND    605
ROGALAND     591
SHETLAND     589
VIKING       475
VESTLAND     200
DUNLIN       257


In [105]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

random_state = 42
test_size = 0.20
val_size = 0.10

# First split: trainval vs test
msss1 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=test_size, random_state=random_state
)
trainval_idx, test_idx = next(msss1.split(wells, Y))

wells_trainval = wells[trainval_idx]
Y_trainval = Y[trainval_idx]
wells_test = wells[test_idx]

# Second split: train vs val
val_size_within_trainval = val_size / (1.0 - test_size)
msss2 = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=val_size_within_trainval, random_state=random_state
)
train_idx2, val_idx = next(msss2.split(wells_trainval, Y_trainval))

train_wells = set(wells_trainval[train_idx2])
val_wells   = set(wells_trainval[val_idx])
test_wells  = set(wells_test)

print(f"\nSplit sizes: train={len(train_wells)}, val={len(val_wells)}, test={len(test_wells)}")


Split sizes: train=439, val=60, test=121


In [106]:
# check_splits function (same as before)
def marker_freq(well_set, well_marker):
    well_sum = well_marker.loc[list(well_set)].sum()
    total = well_sum.sum()
    well_ratio = well_sum / total if total > 0 else well_sum * np.nan
    return pd.concat([well_sum, well_ratio], axis=1, keys=["count", "ratio"])

def check_splits(train_wells, val_wells, test_wells, well_marker, rtol=0.1):
    train_tab = marker_freq(train_wells, well_marker)
    val_tab   = marker_freq(val_wells, well_marker)
    test_tab  = marker_freq(test_wells, well_marker)

    ov_tv = train_wells & val_wells
    ov_tt = train_wells & test_wells
    ov_vt = val_wells & test_wells

    if len(ov_tv) == 0 and len(ov_tt) == 0 and len(ov_vt) == 0:
        print("\nNo overlap between train/val/test wells")
    else:
        print("\nAttention overlap, fix split!")

    close_tv = np.isclose(train_tab["ratio"], val_tab["ratio"], equal_nan=True, rtol=rtol).all()
    close_tt = np.isclose(train_tab["ratio"], test_tab["ratio"], equal_nan=True, rtol=rtol).all()
    close_vt = np.isclose(val_tab["ratio"], test_tab["ratio"], equal_nan=True, rtol=rtol).all()

    print(f"\nRatios close (rtol={rtol}):")
    print("  train vs val :", close_tv)
    print("  train vs test:", close_tt)
    print("  val vs test  :", close_vt)

    print(f"\nSplit sizes (#wells): train={len(train_wells)}, val={len(val_wells)}, test={len(test_wells)}")

    print("\nTrain marker split:\n", train_tab)
    print("\nVal marker split:\n", val_tab)
    print("\nTest marker split:\n", test_tab)

    return train_tab, val_tab, test_tab

check_splits(train_wells, val_wells, test_wells, well_marker)


No overlap between train/val/test wells

Ratios close (rtol=0.1):
  train vs val : False
  train vs test: True
  val vs test  : False

Split sizes (#wells): train=439, val=60, test=121

Train marker split:
            count     ratio
HORDALAND    428  0.222222
ROGALAND     414  0.214953
SHETLAND     419  0.217549
VIKING       345  0.179128
VESTLAND     140  0.072690
DUNLIN       180  0.093458

Val marker split:
            count     ratio
HORDALAND     58  0.222222
ROGALAND      59  0.226054
SHETLAND      60  0.229885
VIKING        38  0.145594
VESTLAND      20  0.076628
DUNLIN        26  0.099617

Test marker split:
            count     ratio
HORDALAND    119  0.224528
ROGALAND     118  0.222642
SHETLAND     110  0.207547
VIKING        92  0.173585
VESTLAND      40  0.075472
DUNLIN        51  0.096226


(           count     ratio
 HORDALAND    428  0.222222
 ROGALAND     414  0.214953
 SHETLAND     419  0.217549
 VIKING       345  0.179128
 VESTLAND     140  0.072690
 DUNLIN       180  0.093458,
            count     ratio
 HORDALAND     58  0.222222
 ROGALAND      59  0.226054
 SHETLAND      60  0.229885
 VIKING        38  0.145594
 VESTLAND      20  0.076628
 DUNLIN        26  0.099617,
            count     ratio
 HORDALAND    119  0.224528
 ROGALAND     118  0.222642
 SHETLAND     110  0.207547
 VIKING        92  0.173585
 VESTLAND      40  0.075472
 DUNLIN        51  0.096226)

In [107]:
LOG_COLS = ["GR_CLEAN", "RHOB_CLEAN", "DEEPRES_CLEAN"]

# df_tops: only filtered wells
df_tops = tops_filtered[MARKERS]

# df_log: all log columns needed
df_log = df_final[[WELL_COL, DEPTH_COL] + LOG_COLS].copy()

# Split logs and tops by well sets
df_log_train = df_log[df_log[WELL_COL].isin(train_wells)]
df_log_valid = df_log[df_log[WELL_COL].isin(val_wells)]
df_log_test  = df_log[df_log[WELL_COL].isin(test_wells)]

df_tops_train = df_tops.loc[df_tops.index.isin(train_wells)]
df_tops_valid = df_tops.loc[df_tops.index.isin(val_wells)]
df_tops_test  = df_tops.loc[df_tops.index.isin(test_wells)]

print(f"df_log_train: {df_log_train.shape}, df_tops_train: {df_tops_train.shape}")
print(f"df_log_valid: {df_log_valid.shape}, df_tops_valid: {df_tops_valid.shape}")
print(f"df_log_test:  {df_log_test.shape},  df_tops_test:  {df_tops_test.shape}")

# All tops wells now have >= 4 tops, so prepare_sequences won't drop any
print(f"\nMin tops per well in train: {df_tops_train[MARKERS].notna().sum(axis=1).min()}")
print(f"Min tops per well in valid: {df_tops_valid[MARKERS].notna().sum(axis=1).min()}")
print(f"Min tops per well in test:  {df_tops_test[MARKERS].notna().sum(axis=1).min()}")

df_log_train: (9510547, 5), df_tops_train: (439, 6)
df_log_valid: (1365031, 5), df_tops_valid: (60, 6)
df_log_test:  (2651036, 5),  df_tops_test:  (121, 6)

Min tops per well in train: 4
Min tops per well in valid: 4
Min tops per well in test:  4


In [108]:
import os, json

os.makedirs("data", exist_ok=True)

df_log_train.to_parquet("data/logs_train.parquet", index=False)
df_log_valid.to_parquet("data/logs_valid.parquet", index=False)
df_log_test.to_parquet("data/logs_test.parquet", index=False)

df_tops_train.to_parquet("data/tops_train.parquet")
df_tops_valid.to_parquet("data/tops_valid.parquet")
df_tops_test.to_parquet("data/tops_test.parquet")

json.dump(list(train_wells), open("data/train_wells.json", "w"))
json.dump(list(val_wells), open("data/val_wells.json", "w"))
json.dump(list(test_wells), open("data/test_wells.json", "w"))

print("Saved all splits to data/")

Saved all splits to data/


In [112]:
'''
for i in range(len(TOP_COLS) - 1):
    upper = TOP_COLS[i]
    lower = TOP_COLS[i + 1]
    both = df_tops_train[[upper, lower]].notna().all(axis=1).sum()
    print(f"  zone {i+1} ({upper}–{lower}): {both} wells with both tops")

# Also zone 0 (needs HORDALAND) and zone 6 (needs DUNLIN)
print(f"  zone 0 (above HORDALAND): {df_tops_train['HORDALAND'].notna().sum()} wells")
print(f"  zone 6 (below DUNLIN):    {df_tops_train['DUNLIN'].notna().sum()} wells")
'''

'\nfor i in range(len(TOP_COLS) - 1):\n    upper = TOP_COLS[i]\n    lower = TOP_COLS[i + 1]\n    both = df_tops_train[[upper, lower]].notna().all(axis=1).sum()\n    print(f"  zone {i+1} ({upper}–{lower}): {both} wells with both tops")\n\n# Also zone 0 (needs HORDALAND) and zone 6 (needs DUNLIN)\nprint(f"  zone 0 (above HORDALAND): {df_tops_train[\'HORDALAND\'].notna().sum()} wells")\nprint(f"  zone 6 (below DUNLIN):    {df_tops_train[\'DUNLIN\'].notna().sum()} wells")\n'